# Do AI models remember Indian company financials?
### Notebook 2 of 2: scoring the answers

**Before you start:** the batched collection is done (`raw_batched.jsonl` has 79 answers per model and prompt), and you've finished the hand check of the 12 audit companies. If a figure in `ground_truth.csv` was wrong, you overwrote it with the company's own number, put the link in `audit_source`, wrote the old Screener value in `audit_note`, and set `audit_done` to 1.

Run one cell at a time and read the text above each one. There's a checkpoint at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, re, math
import numpy as np
import pandas as pd

FOLDER = '/content/drive/MyDrive/llm_memorization_pilot'
truth = pd.read_csv(f'{FOLDER}/ground_truth.csv')
log = pd.DataFrame([json.loads(l) for l in open(f'{FOLDER}/raw_batched.jsonl')])
MODEL_ORDER = ['gpt-oss-120b', 'gemini-3.6-flash', 'gemini-3.5-flash-lite']   # main Gemini = 3.6 Flash; Flash-Lite = smaller sibling
log = log[log['model'].isin(MODEL_ORDER)]
log = log[log['status'] == 'ok'].sort_values('ts').drop_duplicates(['model', 'variant', 'code'], keep='last')
print(len(truth), 'companies |', len(log), 'responses')
print('audit done for', int(truth['audit_done'].sum()), 'of', int((truth['audit'].fillna('') != '').sum()), 'audit companies')

## Step 3a: from raw text to a tidy table

Right now each response is one blob of JSON text holding 6 numbers (2 metrics × 3 years). For analysis we want **one row per number**: model, prompt, company, year, metric, the model's answer and the true value. This is called *long format*. Almost every analysis in pandas becomes a one-liner once data is shaped like this.

The parser is deliberately forgiving about *format*: "2,40,893" becomes 240893, and keys like "fy2024" become FY24. It is never forgiving about *content*: a missing number stays missing.

In [ ]:
def parse_json(text):
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text or '', re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: return None
    return None

def to_number(x):
    if x is None or isinstance(x, bool): return None
    if isinstance(x, (int, float)): return float(x) if math.isfinite(x) else None
    s = re.sub(r'[^\d.\-]', '', str(x))          # drop commas, "₹", "crore" etc.
    try: return float(s) if s not in ('', '-', '.') else None
    except ValueError: return None

def year_key(k):
    for y in ('23', '24', '25', '26'):
        if y in str(k): return 'FY' + y
    return None

rows = []
for _, r in log.iterrows():
    d = parse_json(r['raw']) or {}
    by_year = {year_key(k): v for k, v in d.items() if year_key(k) and isinstance(v, dict)}
    for fy in ('FY23', 'FY24', 'FY25', 'FY26'):
        v = {str(k).lower(): val for k, val in by_year.get(fy, {}).items()}
        conf = to_number(v.get('confidence'))
        for metric, keys in (('revenue', ('revenue', 'revenue_from_operations', 'sales')),
                             ('net_profit', ('net_profit', 'netprofit', 'profit', 'pat'))):
            pred = next((to_number(v[k]) for k in keys if k in v), None)
            rows.append({'model': r['model'], 'variant': r['variant'], 'code': r['code'],
                         'year': fy, 'metric': metric, 'pred': pred, 'confidence': conf,
                         'parsed': bool(d)})

long = pd.DataFrame(rows)
col = {(m, f'FY{y}'): f'{p}_fy{y}' for m, p in (('revenue', 'rev'), ('net_profit', 'np')) for y in (23, 24, 25, 26)}
t = truth.set_index('code')
long['truth'] = [t.loc[c, col[(m, y)]] for c, m, y in zip(long['code'], long['metric'], long['year'])]
long = long.merge(truth[['code', 'company', 'tier', 'mcap_cr']], on='code')
long = long[long['truth'].notna()]      # Linde India and Owais Metal have no usable FY23 figure: drop, don't count as misses
print(long.shape)
long.head(6)

## Step 3b: how wrong is each answer?

**Absolute percentage error (APE)** = |answer − truth| ÷ |truth|.
Example: a model says TCS FY24 revenue was ₹2,40,000 cr, the truth is ₹2,40,893 cr, so APE = 893 ÷ 2,40,893 = 0.37%.

- **hit5:** APE ≤ 5%. This is the headline measure: "the model basically knew the number".
- **hit1:** APE ≤ 1%. Near-exact recall, the strongest sign of memorization.
- **Unit error:** the answer is off by almost exactly 10×, 100× or 1000× (₹ million or ₹ lakh instead of ₹ crore), or looks like US$ million (about 0.11× the crore figure). We count these separately rather than silently fixing them. A model that knows the number but in the wrong unit is still a memorization signal, and we report it as a robustness check.

Why 5%? Revenue figures get small revisions and definition differences (gross vs net of some items). 5% is tight enough that a lucky guess is rare and loose enough to forgive bookkeeping noise. We also report 1% and the median APE, so nothing hinges on one threshold.

In [ ]:
long['answered'] = long['pred'].notna()
long['ape'] = (long['pred'] - long['truth']).abs() / long['truth'].abs()
long['hit5'] = long['ape'] <= 0.05          # NaN (no answer) compares as False: a non-answer is a miss
long['hit1'] = long['ape'] <= 0.01

def unit_error(pred, truth):
    if pred is None or not np.isfinite(pred) or truth == 0 or pred == 0: return None
    ratio = abs(pred / truth)
    for k, name in ((10, 'x10 (INR million?)'), (100, 'x100 (INR lakh?)'), (0.1, 'x0.1'), (0.01, 'x0.01'),
                    (1000, 'x1000'), (0.001, 'x0.001')):
        if abs(ratio / k - 1) <= 0.07: return name
    if 0.105 <= ratio <= 0.125: return 'USD million?'
    return None

long['unit_error'] = [unit_error(p, tr) for p, tr in zip(long['pred'], long['truth'])]
print('Answers with a unit error:', long['unit_error'].notna().sum())
print(long['unit_error'].value_counts())

# Robustness version: if the only problem is the unit, rescale and re-score.
scale = {'x10 (INR million?)': 10, 'x100 (INR lakh?)': 100, 'x0.1': 0.1, 'x0.01': 0.01, 'x1000': 1000, 'x0.001': 0.001}
fixed = long['pred'] / long['unit_error'].map(scale).fillna(1)
long['hit5_unitfixed'] = ((fixed - long['truth']).abs() / long['truth'].abs()) <= 0.05

## Step 3c: label each (model, year) as before or after the model's training cutoff

| | FY23 (results Apr–May 2023) | FY24 (Apr–May 2024) | FY25 (Apr–May 2025) | FY26 (Apr–May 2026) |
|---|---|---|---|---|
| gpt-oss-120b (cutoff 1 Jun 2024) | before | edge | after | after |
| gemini-3.6-flash (cutoff Mar 2026) | before | before | before | after |
| gemini-3.5-flash-lite (cutoff Mar 2026, smaller model) | before | before | before | after |

"After" cells are the **control group**: the model *cannot* have seen those numbers, so any hits there come from guessing well (for example, assuming steady growth from a year it does know).

In [ ]:
CUTOFF = {('gpt-oss-120b', 'FY23'): 'before', ('gpt-oss-120b', 'FY24'): 'edge',
          ('gpt-oss-120b', 'FY25'): 'after', ('gpt-oss-120b', 'FY26'): 'after',
          **{(g, fy): ('after' if fy == 'FY26' else 'before')
             for g in ('gemini-3.6-flash', 'gemini-3.5-flash-lite') for fy in ('FY23', 'FY24', 'FY25', 'FY26')}}
long['cutoff'] = [CUTOFF[(m, y)] for m, y in zip(long['model'], long['year'])]
TIERS = ['1_Large', '2_Mid', '3_Small', '4_Micro']
long.groupby(['model', 'year'])['cutoff'].first().unstack()

## Result 1: does the model know what it doesn't know? (strict prompt)

Share of figures where the model gave a number instead of null. A well-behaved model answers more for big companies and before its cutoff, and almost never for years after its cutoff. **Any answer in an "after" year is a confident guess presented as fact**, which matters for anyone using these models for research.

In [ ]:
strict = long[(long['variant'] == 'strict') & (long['metric'] == 'revenue')]
r1 = strict.pivot_table(index=['model', 'tier'], columns='year', values='answered', aggfunc='mean').round(2)
r1

## Result 2 (the main table): hit rate by size tier, year and model (best-estimate prompt, revenue)

Each cell is the share of companies where the model's revenue answer was within 5% of the truth. The brackets are **95% bootstrap confidence intervals**: we re-draw the set of companies 2,000 times with replacement and see how much the hit rate moves. With 14–28 companies per tier the intervals are wide, and that is honest: this is a pilot.

**The random-walk row** is a dumb benchmark: "this year's revenue = last year's true revenue". It hits whenever a company's revenue moved less than 5%. If a model's "after cutoff" hit rate is close to this, it's guessing from trends, not remembering. (FY23 has no random-walk value because our data starts at FY23.)

In [ ]:
rng = np.random.default_rng(2026)

def boot_ci(x, n=2000):
    x = np.asarray(x, dtype=float)
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    draws = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return x.mean(), np.percentile(draws, 2.5), np.percentile(draws, 97.5)

best_rev = long[(long['variant'] == 'best') & (long['metric'] == 'revenue')]

def cell(df):
    m, lo, hi = boot_ci(df['hit5'])
    return f'{m:.0%} [{lo:.0%}–{hi:.0%}]'

main = (best_rev.groupby(['model', 'tier', 'year']).apply(cell, include_groups=False)
        .unstack('year').reindex(columns=['FY23', 'FY24', 'FY25', 'FY26']))
overall = best_rev.groupby(['model', 'year']).apply(cell, include_groups=False).unstack('year')
overall.index = pd.MultiIndex.from_tuples([(m, 'ALL') for m in overall.index])

# Random-walk benchmark: predict this year = last year's true value
rw = truth[['tier']].copy()
for prev, cur in ((23, 24), (24, 25), (25, 26)):
    ok = truth[f'rev_fy{prev}'].notna()
    rw[f'rw_FY{cur}'] = ((truth[f'rev_fy{prev}'] / truth[f'rev_fy{cur}'] - 1).abs() <= 0.05).where(ok)
rw_rows = rw.groupby('tier')[['rw_FY24', 'rw_FY25', 'rw_FY26']].mean()
rw_tab = pd.DataFrame({'FY23': '—', **{c[3:]: rw_rows[c].map('{:.0%}'.format) for c in rw_rows.columns}})
rw_tab.index = pd.MultiIndex.from_tuples([('random walk', t) for t in rw_tab.index])

main_table = pd.concat([main, overall, rw_tab]).sort_index()
main_table.to_csv(f'{FOLDER}/table_main_hit5_revenue.csv')
main_table

### Two more views of the same cells
- **Median APE**: how far off the typical answer is (lower = better).
- **hit1**: share within 1%, the near-exact recall that's hard to get by guessing.

In [ ]:
med = best_rev.pivot_table(index=['model', 'tier'], columns='year', values='ape', aggfunc='median').map(lambda v: f'{v:.1%}')
h1 = best_rev.pivot_table(index=['model', 'tier'], columns='year', values='hit1', aggfunc='mean').map(lambda v: f'{v:.0%}')
display(med); display(h1)

## Result 3: the formal test (logistic regression)

Tables show patterns. A regression asks: **holding everything else fixed, does company size still matter, and does being before the cutoff still matter?**

Model: P(hit5) depends on log(market cap), the cutoff status (before / edge / after) and which AI model it is.
- A **positive coefficient on `log_mcap`** means bigger companies are recalled better.
- A **positive coefficient on `cutoff[T.before]`** (compared with "after") is memorization: knowing numbers it could only have seen in training.

Each company appears 6 times (2 models × 3 years), so its answers are not independent draws. **Clustered standard errors by company** correct for that. Say this sentence if a professor asks about inference.

In [ ]:
import statsmodels.formula.api as smf
reg = best_rev.assign(log_mcap=np.log(best_rev['mcap_cr']), hit=best_rev['hit5'].astype(int))
try:
    fit = smf.logit("hit ~ log_mcap + C(cutoff, Treatment('after')) + C(model)", data=reg).fit(
        disp=0, cov_type='cluster', cov_kwds={'groups': pd.factorize(reg['code'])[0]})
    print(fit.summary2().tables[1][['Coef.', 'Std.Err.', 'P>|z|']].round(3))
    print('\nAverage marginal effects (change in hit probability):')
    print(fit.get_margeff().summary_frame()[['dy/dx', 'Pr(>|z|)']].round(3))
except Exception as e:
    print('Regression failed (usually a cell with all hits or all misses):', e)

## Result 4: does the model's confidence mean anything?

The best-estimate prompt asked for a 0–100 confidence. If confidence is meaningful, answers given at 80+ should be right far more often than answers at 0–40.

In [ ]:
cal = best_rev.dropna(subset=['confidence']).copy()
cal['conf_bin'] = pd.cut(cal['confidence'], [-1, 40, 60, 80, 100], labels=['0–40', '41–60', '61–80', '81–100'])
cal.pivot_table(index='conf_bin', columns='model', values='hit5', aggfunc=['mean', 'count'], observed=False).round(2)

## The two figures for the paper

**Figure 1:** hit rate by size tier, one panel per model, one line per fiscal year. If memorization is real, the lines for years before each model's cutoff sit high on the left (large caps) and slope down to the right, while lines for "after" years sit flat near the bottom.

**Figure 2:** the model's answer vs the true revenue for FY23, a year every model could have seen, on log scales. Points on the dashed diagonal are exact recall.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D

MODEL_ORDER = ['gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gpt-oss-120b']
MODEL_LABEL = {'gemini-3.6-flash': 'Gemini 3.6 Flash (cutoff Mar 2026)',
               'gemini-3.5-flash-lite': 'Gemini 3.5 Flash-Lite (cutoff Mar 2026)',
               'gpt-oss-120b': 'gpt-oss-120b (cutoff Jun 2024)'}
CUTOFF_NOTE = {'gemini-3.6-flash': 'FY23–FY25 before cutoff, FY26 after',
               'gemini-3.5-flash-lite': 'FY23–FY25 before cutoff, FY26 after',
               'gpt-oss-120b': 'FY23 before, FY24 edge, FY25–FY26 after'}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
YEAR_COLOR = {'FY23': '#2a78d6', 'FY24': '#eb6834', 'FY25': '#1baf7a', 'FY26': '#eda100'}
YEAR_MARK = {'FY23': 'o', 'FY24': 's', 'FY25': '^', 'FY26': 'D'}
TIER_LABEL = {'1_Large': 'Large', '2_Mid': 'Mid', '3_Small': 'Small', '4_Micro': 'Micro'}
plt.rcParams.update({'font.size': 10, 'axes.edgecolor': INK2, 'axes.labelcolor': INK2, 'xtick.color': INK2,
                     'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False})

rates = best_rev.groupby(['model', 'year', 'tier'])['hit5'].mean()
models = [m for m in MODEL_ORDER if m in set(best_rev['model'])]
fig, axes = plt.subplots(1, len(models), figsize=(4.5 * len(models), 3.9), sharey=True, squeeze=False)
axes = axes[0]
for ax, m in zip(axes, models):
    for fy in ['FY23', 'FY24', 'FY25', 'FY26']:
        y = [rates.get((m, fy, t), np.nan) for t in TIERS]
        ax.plot(range(4), y, color=YEAR_COLOR[fy], lw=2, marker=YEAR_MARK[fy], ms=7, mec='white', mew=1.5, clip_on=False)
    ax.set_title(MODEL_LABEL[m], color=INK, fontsize=10.5, loc='left', pad=20)
    ax.text(0, 1.015, CUTOFF_NOTE[m], transform=ax.transAxes, fontsize=8.5, color=INK2, va='bottom')
    ax.set_xticks(range(4), [TIER_LABEL[t] for t in TIERS])
    ax.set_xlabel('Company size tier (market cap)')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_ylim(0, 1.04); ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.grid(axis='y', color=GRID, lw=0.8); ax.set_axisbelow(True)
axes[0].set_ylabel('Revenue answers within 5% of truth')
handles = [Line2D([], [], color=YEAR_COLOR[fy], lw=2, marker=YEAR_MARK[fy], ms=7, mec='white', mew=1.5, label=fy)
           for fy in ['FY23', 'FY24', 'FY25', 'FY26']]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(f'{FOLDER}/fig1_hit_rate_by_tier.png', dpi=300)
plt.show()


In [ ]:
fy23 = best_rev[(best_rev['year'] == 'FY23') & best_rev['pred'].notna() & (best_rev['pred'] > 0)]
fig, axes = plt.subplots(1, len(models), figsize=(4.5 * len(models), 4.2), sharex=True, sharey=True, squeeze=False)
axes = axes[0]
lims = (fy23[['pred', 'truth']].min().min() * 0.5, fy23[['pred', 'truth']].max().max() * 2)
for ax, m in zip(axes, models):
    d = fy23[fy23['model'] == m]
    ax.plot(lims, lims, ls='--', color=INK2, lw=1, zorder=1)
    ax.scatter(d['truth'], d['pred'], s=36, color='#2a78d6', edgecolor='white', linewidth=1, zorder=2)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(MODEL_LABEL[m].split(' (')[0] + ': FY23 revenue', color=INK, fontsize=10.5, loc='left')
    ax.set_xlabel('Reported revenue (INR crore, log scale)')
    ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
    ax.text(0.03, 0.95, f"within 5%: {d['hit5'].mean():.0%} of {len(d)}", transform=ax.transAxes, color=INK2, va='top')
axes[0].set_ylabel("Model's answer (INR crore, log scale)")
fig.tight_layout()
fig.savefig(f'{FOLDER}/fig2_pred_vs_true_fy23.png', dpi=300)
plt.show()


## Robustness checks (for the paper's appendix)
1. **Net profit** instead of revenue (expect lower hit rates: definitions vary, and small profits make % errors explode).
2. **Unit-corrected** hit rate.
3. **Excluding the two suspect companies** (Vedanta, Coal India), in case their ground truth is still off.

In [ ]:
best_np = long[(long['variant'] == 'best') & (long['metric'] == 'net_profit')]
checks = pd.DataFrame({
    'revenue hit5 (main)': best_rev.groupby(['model', 'year'])['hit5'].mean(),
    'net profit hit5': best_np.groupby(['model', 'year'])['hit5'].mean(),
    'revenue hit5, unit-fixed': best_rev.groupby(['model', 'year'])['hit5_unitfixed'].mean(),
    'revenue hit5, excl. suspects': best_rev[~best_rev['code'].isin(['VEDL', 'COALINDIA'])].groupby(['model', 'year'])['hit5'].mean(),
}).round(2)
checks.to_csv(f'{FOLDER}/table_robustness.csv')
checks

In [ ]:
# Save the tidy table: it becomes the public dataset on GitHub
long.to_csv(f'{FOLDER}/scored_answers_long.csv', index=False)
print('saved', len(long), 'rows')

## ✋ CHECKPOINT 2

Send me the outputs of **Result 2, Result 3 and the robustness table**, plus your answers (2–3 lines each, your own words):

1. Read one cell of the main table out loud as a sentence a professor would understand. Example shape: "For large caps, Gemini got FY23 revenue within 5% for X% of companies."
2. Why do we re-draw *companies* in the bootstrap rather than individual answers?
3. gpt-oss scores some hits in FY25, after its cutoff. Give two explanations that aren't memorization, and say which table row helps you tell them apart.
4. What does the coefficient on `log_mcap` tell you, in plain words?
5. Which single number from all of this goes in the first line of your email, and why that one?